In [23]:
import json
import random

random.seed(42)

dataset_with_answers_path = "musique_ans_v1.0_dev.jsonl"
examples_with_answers = []

all_lines = []
with open(dataset_with_answers_path, 'r', encoding='utf-8') as f:
    all_lines = [json.loads(line) for line in f]


examples_with_answers = random.sample(all_lines, 700)[300:400]

print(f"download {len(examples_with_answers)} случайных примеров")

download 100 случайных примеров


In [24]:
import pandas as pd

data = examples_with_answers

rows = []
for item in data:
 
    q_id = item['id']
    question = item['question']
    
    support_idxs = [dec['paragraph_support_idx'] for dec in item.get('question_decomposition', [])]
    
    rows.append({
        'id': q_id,
        'question': question,
        'paragraph_support_idx': support_idxs  # список
    })

main_question_df = pd.DataFrame(rows)
main_question_df

,id,question,paragraph_support_idx
0,2hop__81825_49084,Who plays the character that took the sword ou...,"[10, 0]"
1,2hop__131455_11960,What is the tallest building in the state wher...,"[8, 19]"
2,3hop1__694534_160088_85460,What is the average salary of a working person...,"[17, 1, 13]"
3,2hop__726717_610238,What is the network which National Cycle Route...,"[12, 14]"
4,3hop1__106864_160713_77246,What is the meaning in the Arabic Dictionary o...,"[12, 8, 7]"
...,...,...,...
95,2hop__1667_40501,How many refugees emigrated to the nation that...,"[7, 10]"
96,2hop__61714_53995,When do Pam and Pam's spouse on The Office end...,"[0, 9]"
97,4hop1__146971_698949_157828_162309,When did the country that has the same co-offi...,"[14, 8, 18, 10]"
98,2hop__274110_413723,In which district was Ernie Watts born?,"[14, 18]"


In [25]:
import pandas as pd
from typing import List, Dict, Any
import re

def transform_dataset(dataset: List[Dict[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame]:

    paragraphs_list = []
    questions_list = []
    global_paragraph_counter = 1
    paragraph_mapping = {}

    for example in dataset:
        example_id = example['id']

        for paragraph in example['paragraphs']:
            paragraph_idx = paragraph['idx']
            key = (example_id, paragraph_idx)


            paragraph_mapping[key] = global_paragraph_counter


            paragraphs_list.append({
                'absolute_id': global_paragraph_counter,
                'text': paragraph['paragraph_text']
            })

            global_paragraph_counter += 1


    for example in dataset:
        example_id = example['id']
        main_question_id = example['id']
        main_question_text = example['question']

        decomposition = example.get('question_decomposition', [])


        processed_questions = []

        for i, sub_question in enumerate(decomposition):
            question_text = sub_question['question']


            answer_mapping = {}
            for j in range(i):
                if j < len(processed_questions):
                    placeholder = f"#{j+1}"
                    answer_mapping[placeholder] = decomposition[j].get('answer', '')


            for placeholder, answer in answer_mapping.items():
                if answer:

                    #print(question_text)
                    pattern = re.escape(placeholder)
                    #print(pattern)

                    question_text = re.sub(placeholder, answer, question_text)
                    #print(question_text)

            processed_questions.append(question_text)


        for i, sub_question in enumerate(decomposition):
            paragraph_support_idx = sub_question.get('paragraph_support_idx')


            paragraph_absolute_id = None
            if paragraph_support_idx is not None:
                key = (example_id, paragraph_support_idx)
                paragraph_absolute_id = paragraph_mapping.get(key)

            questions_list.append({
                'question': processed_questions[i],
                'paragraph_absolute_id': paragraph_absolute_id,
                'main_question_id': main_question_id,
                'main_question_text': main_question_text
            })


    paragraphs_df = pd.DataFrame(paragraphs_list)
    questions_df = pd.DataFrame(questions_list)

    return paragraphs_df, questions_df

In [26]:
paragraphs_df, questions_df = transform_dataset(examples_with_answers)
paragraphs_df

,absolute_id,text
0,1,Liam Thomas Garrigan (born 17 October 1981) is...
1,2,Ideas for a Conan film were proposed as early ...
2,3,"Jeffrey Shawn Swords (born December 27, 1973 i..."
3,4,``Born in the U.S.A. ''is a 1984 song written ...
4,5,"Haji Sahib of Turangzai, the most famous Pukht..."
...,...,...
1993,1994,"In the earlier seasons of Family Guy, Clevelan..."
1994,1995,Lacey Chabert voiced Meg for the first product...
1995,1996,Meg Griffin Family Guy character First appeara...
1996,1997,John Herbert Family Guy character First appear...


In [27]:
paragraph_support_idx = []
for main_question_id in main_question_df.id:
    main_question_support_idx = list(questions_df[questions_df['main_question_id'] == main_question_id].paragraph_absolute_id)
    paragraph_support_idx.append(main_question_support_idx)
main_question_df['paragraph_support_idx'] = paragraph_support_idx
main_question_df

,id,question,paragraph_support_idx
0,2hop__81825_49084,Who plays the character that took the sword ou...,"[11, 1]"
1,2hop__131455_11960,What is the tallest building in the state wher...,"[29, 40]"
2,3hop1__694534_160088_85460,What is the average salary of a working person...,"[58, 42, 54]"
3,2hop__726717_610238,What is the network which National Cycle Route...,"[73, 75]"
4,3hop1__106864_160713_77246,What is the meaning in the Arabic Dictionary o...,"[93, 89, 88]"
...,...,...,...
95,2hop__1667_40501,How many refugees emigrated to the nation that...,"[1906, 1909]"
96,2hop__61714_53995,When do Pam and Pam's spouse on The Office end...,"[1919, 1928]"
97,4hop1__146971_698949_157828_162309,When did the country that has the same co-offi...,"[1953, 1947, 1957, 1949]"
98,2hop__274110_413723,In which district was Ernie Watts born?,"[1973, 1977]"


In [28]:
import numpy as np
import re
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


class BM25Retriever:

    def __init__(
        self,
        paragraphs_df,
        questions_df=None,
        lowercase=True,
        remove_stopwords=True,
    ):

        self.paragraphs_df = paragraphs_df
        self.questions_df = questions_df

        self.lowercase = lowercase
        self.remove_stopwords = remove_stopwords

        self.ids = paragraphs_df["absolute_id"].astype(str).tolist()
        self.texts = paragraphs_df["text"].fillna("").astype(str).tolist()

        self.tokenized_corpus = [
            self._tokenize(text)
            for text in self.texts
        ]

        self.bm25 = BM25Okapi(self.tokenized_corpus)


    # Tokenization 
    def _tokenize(self, text: str):

        if self.lowercase:
            text = text.lower()

        tokens = re.findall(r"\w+", text)

        if self.remove_stopwords:
            tokens = [
                t for t in tokens
                if t not in ENGLISH_STOP_WORDS
            ]

        return tokens


    # Retrieval
    def retrieve(self, query, topk=10):

        query_tokens = self._tokenize(query)

        scores = self.bm25.get_scores(query_tokens)

        top_idx = np.argpartition(-scores, topk)[:topk]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        return [
            {
                "id": self.ids[i],
                "score": float(scores[i])
            }
            for i in top_idx
        ]


    # Evaluation
    def evaluate(self, topk=10):

        if self.questions_df is None:
            raise ValueError("questions_df is None")

        total_p, total_r, total_f1 = 0, 0, 0

        for _, row in tqdm(
            self.questions_df.iterrows(),
            total=len(self.questions_df)
        ):

            gold = set(map(str, row["paragraph_support_idx"]))

            results = self.retrieve(row["question"], topk=topk)

            pred = [r["id"] for r in results]
            pred_set = set(pred)

            tp = len(pred_set & gold)

            precision = tp / len(pred) if pred else 0
            recall = tp / len(gold) if gold else 0

            f1 = (
                2 * precision * recall / (precision + recall)
                if (precision + recall) else 0
            )

            total_p += precision
            total_r += recall
            total_f1 += f1

        n = len(self.questions_df)

        return {
            "precision": total_p / n,
            "recall": total_r / n,
            "f1": total_f1 / n
        }


    # Utility
    def get_top_k_relevant(self, query, k=5, page=0):

        # --- tokenization (лучше чем split)
        tokenized_query = self._tokenize(query)

        # --- BM25 scores
        scores = self.bm25.get_scores(tokenized_query)

        # --- top sorting (fast version)
        sorted_indices = sorted(
            range(len(scores)),
            key=lambda i: scores[i],
            reverse=True
        )

        # --- pagination
        start = page * k
        end = start + k
        top_indices = sorted_indices[start:end]

        # --- results
        top_ids = [int(self.ids[i]) for i in top_indices]
        top_texts = [self.texts[i] for i in top_indices]

        docs = [
            f"{pid}. {txt}"
            for pid, txt in zip(top_ids, top_texts)
        ]

        return {
            "docs": docs,
            "ids": top_ids
        }


In [29]:
retriever = BM25Retriever(
    paragraphs_df=paragraphs_df,
    questions_df=main_question_df
)

In [11]:
import random
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

class Retriever:
    def __init__(self, qa_df, paragraphs_df, top_k=5, p_correct=0.5, seed=None,
                 model_name='all-MiniLM-L6-v2', device='cuda'):
        if seed is not None:
            random.seed(seed)
            torch.manual_seed(seed)

        self.top_k = top_k
        self.p_correct = p_correct
        self.device = device

        self.encoder = SentenceTransformer(model_name, device=device)

        self.paragraph_ids = paragraphs_df['absolute_id'].tolist()
        self.paragraph_texts = paragraphs_df['text'].tolist()
        print("Calculating paragraph embeddings...")
        self.paragraph_embeddings = self.encoder.encode(
            self.paragraph_texts, convert_to_tensor=True, device=device, show_progress_bar=True
        )
        self.paragraph_embeddings = torch.nn.functional.normalize(self.paragraph_embeddings, p=2, dim=1)

        self.correct_map = {}
        for _, row in qa_df.iterrows():
            q = row['question']
            pid = row['paragraph_absolute_id']
            self.correct_map.setdefault(q, set()).add(pid)

        self.text_by_id = dict(zip(self.paragraph_ids, self.paragraph_texts))

    def get_top_k_relevant(self, query, k=5, page=0):
        query_emb = self.encoder.encode(query, convert_to_tensor=True, device=self.device)
        query_emb = torch.nn.functional.normalize(query_emb, p=2, dim=0)
        similarities = torch.matmul(self.paragraph_embeddings, query_emb)
        sorted_indices = torch.argsort(similarities, descending=True).cpu().numpy()
        sorted_ids = [self.paragraph_ids[i] for i in sorted_indices]
        start = page * k
        end = start + k
        top_ids = sorted_ids[start:end]
        top_texts = [self.text_by_id[pid] for pid in top_ids]
        docs = [f"{pid}. {txt}" for pid, txt in zip(top_ids, top_texts)]

        return {
            'docs': docs,
            'ids': top_ids
        }

    def __call__(self, query, last):
        query_emb = self.encoder.encode(query, convert_to_tensor=True, device=self.device)
        query_emb = torch.nn.functional.normalize(query_emb, p=2, dim=0)
        similarities = torch.matmul(self.paragraph_embeddings, query_emb)
        sorted_indices = torch.argsort(similarities, descending=True).cpu().numpy()
        sorted_ids = [self.paragraph_ids[i] for i in sorted_indices]

        correct_ids = self.correct_map.get(query, set())
        candidates = sorted_ids[:self.top_k]

        if last == False:
            include_correct = random.random() < self.p_correct
        else:
            include_correct = True

        correct_in_candidates = [pid for pid in candidates if pid in correct_ids]

        if include_correct:
            if not correct_in_candidates and correct_ids:
                chosen_correct = random.choice(list(correct_ids))
                replace_idx = random.randint(0, len(candidates) - 1)
                candidates[replace_idx] = chosen_correct
        else:
            if correct_in_candidates:
                candidates_without_correct = [pid for pid in candidates if pid not in correct_ids]
                next_ids = []
                for pid in sorted_ids[self.top_k:]:
                    if pid not in correct_ids and pid not in candidates_without_correct:
                        next_ids.append(pid)
                    if len(candidates_without_correct) + len(next_ids) >= self.top_k:
                        break
                candidates = (candidates_without_correct + next_ids)[:self.top_k]

        random.shuffle(candidates)

        docs = [f"{pid}. {self.text_by_id[pid]}" for pid in candidates]
        ids = [pid for pid in candidates]

        return {
            'docs': docs,
            'ids': ids
        }

In [12]:
TOP_K = 5
SEED = 42
RETRIEVER_MODEL = 'Qwen3-Embedding-4B'
DEVICE = "cuda"

qwen_retriever = Retriever(questions_df, paragraphs_df, top_k=TOP_K,
                             seed=SEED, model_name=RETRIEVER_MODEL, device=DEVICE)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Вычисление эмбеддингов параграфов...


Batches:   0%|          | 0/58 [00:00<?, ?it/s]

In [13]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "submit_answer",
            "description": "Submit the final answer with the document ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "Document ID that best answers the question"}
                },
                "required": ["id"]
            }
        }
    }
]

In [14]:
import json
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Dict, Any, Optional, Tuple, Union


class QwenAgent:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-7B-Instruct", device_map: str = "auto"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map=device_map
        )
        self.model.eval()

    def _call_model(self, messages: List[Dict[str, Any]], max_new_tokens: int = 2048):

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tools=TOOLS,
            add_generation_prompt=True,
            tokenize=False
        )
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0,
                do_sample=False,
                return_dict_in_generate=True,
                output_scores=True,
            )
        # generated_ids: [1, total_seq_len]; входные + новые
        total_ids = outputs.sequences[0]
        input_len = inputs.input_ids.shape[1]
        generated_ids = total_ids[input_len:]  # только новые токены
        scores = outputs.scores  # tuple of tensors, каждый [1, vocab_size]
        response = self.tokenizer.decode(generated_ids, skip_special_tokens=False)
        return response, generated_ids, scores

    import json
    import re
    from typing import Optional, Dict, Any


    def _extract_tool_call(self, response: str) -> Optional[Dict[str, Any]]:
        if not response:
            return None

        block_match = re.search(r"<tool_call>(.*?)</tool_call>", response, re.DOTALL)
        if not block_match:
            return None

        content = block_match.group(1)


        name = None

        name_match = re.search(r"function\s*=\s*([a-zA-Z0-9_]+)", content)
        if name_match:
            name = name_match.group(1)

        if not name:
            name_match = re.search(r'"name"\s*:\s*"([^"]+)"', content)
            if name_match:
                name = name_match.group(1)

        if not name:
            return None


        args = {}

        # try valid JSON arguments block first
        args_match = re.search(r'"arguments"\s*:\s*(\{.*\})', content, re.DOTALL)

        if args_match:
            raw_args = args_match.group(1)

            # fix common LLM issues
            raw_args = raw_args.strip()
            raw_args = re.sub(r"[}>]+\s*$", "}", raw_args)

            try:
                args = json.loads(raw_args)
            except Exception:
                args = {}


        if not args:
            param_match = re.search(r'<parameter=ids>\s*(\[.*?\])\s*</parameter>', content, re.DOTALL)
            if param_match:
                try:
                    ids_list = json.loads(param_match.group(1))
                    if isinstance(ids_list, list):
                        args = {"ids": [int(x) for x in ids_list]}
                except:
                    pass


        if not args:
            ids_match = re.search(r'"ids"\s*:\s*\[([0-9,\s]+)\]', content)
            if ids_match:
                ids = [
                    int(x)
                    for x in ids_match.group(1).split(",")
                    if x.strip().isdigit()
                ]
                args = {"ids": ids}

        return {
            "name": name,
            "arguments": args,
        }

    def _compute_tool_metric(self, response: str, generated_ids: torch.Tensor, scores: tuple, metric: str = 'entropy') -> Optional[float]:

        tool_call_pattern = r'(<tool_call>.*?</tool_call>)'
        match = re.search(tool_call_pattern, response, re.DOTALL)
        if not match:
            return None
        tool_call_str = match.group(1)
        start_idx = match.start()
        end_idx = match.end()


        tokens = []
        positions = []
        pos = 0
        for tid in generated_ids:
            token_str = self.tokenizer.decode([tid], skip_special_tokens=False)
            tokens.append(token_str)
            positions.append(pos)
            pos += len(token_str)

        full_text = ''.join(tokens)


        indices = []
        for i, (token, p) in enumerate(zip(tokens, positions)):
            token_end = p + len(token)
            if token_end > start_idx and p < end_idx:
                indices.append(i)

        if not indices:
            return None

        values = []
        for i in indices:
            logits = scores[i]          # [1, vocab_size]
            logp = F.log_softmax(logits, dim=-1)   # [1, vocab_size]
            if metric == 'entropy':
                p = logp.exp()
                entropy = -(p * logp).sum(dim=-1).item()
                values.append(entropy)
            elif metric == 'nll':
                token_id = generated_ids[i].item()
                nll = -logp[0, token_id].item()
                values.append(nll)
            else:
                raise ValueError("metric must be 'entropy' or 'nll'")

        return sum(values) / len(values)

    def step(self, messages: List[Dict[str, Any]], metric: str = 'entropy') -> Tuple[Optional[Dict[str, Any]], str, Optional[float]]:
        #print('hello')
        response, gen_ids, scores = self._call_model(messages)
        tool_call = self._extract_tool_call(response)
        tool_metric = -1 #self._compute_tool_metric(response, gen_ids, scores, metric) if tool_call else None
        return tool_call, response, tool_metric

In [15]:
import re
import torch
import torch.nn.functional as F

# ======================== SYSTEM PROMPT ========================
SYSTEM_PROMPT = """
ROLE:
You are a retrieval QA agent.

TASK:
Given a question and a list of numbered documents, identify the minimal set of documents required to answer the question.

IMPORTANT:
Your ONLY goal is to select supporting document ids.

BEHAVIOR RULES:
1. NEVER generate long reasoning chains.
2. NEVER explain the final answer.
3. NEVER summarize documents.
4. NEVER repeat document text.
5. NEVER think step-by-step publicly.
6. Use at most ONE short <thought> block.
7. The <thought> block must be under 20 words.
8. After the thought, immediately output the tool call.
9. DO NOT output anything after the tool call.
10. NEVER skip the tool call.
11. Output exactly ONE tool call.

DOCUMENT SELECTION RULES:
1. Select the SMALLEST sufficient set of document ids.
2. Some questions require combining several documents.
3. Ignore irrelevant or redundant documents.
4. If evidence is insufficient, return empty ids array.
5. NEVER invent facts or ids.
6. Prefer explicit evidence over inferred assumptions.

OUTPUT FORMAT:

<thought>short reasoning</thought>
<tool_call>
{"name":"submit_answer","arguments":{"ids":[1,2]}}
</tool_call>

EXAMPLES:

EXAMPLE 1 (multi-hop acquisition + CEO):
Question: Who was the CEO of the company that acquired YouTube?

Documents:
4. YouTube was acquired by Google in 2006.
18. Sundar Pichai is the CEO of Google.
29. YouTube is a video-sharing platform.
51. Alphabet owns Google.

<thought>Need acquiring company and CEO.</thought>
<tool_call>
{"name":"submit_answer","arguments":{"ids":[4,18]}}
</tool_call>


EXAMPLE 2 (temporal multi-hop):
Question: Which U.S. president was in office when the first iPhone was released?

Documents:
6. The first iPhone was released in 2007.
15. George W. Bush served as U.S. president from 2001 to 2009.
21. Barack Obama became president in 2009.
44. Apple introduced the App Store in 2008.

<thought>Need release year and presidency period.</thought>
<tool_call>
{"name":"submit_answer","arguments":{"ids":[6,15]}}
</tool_call>


EXAMPLE 3 (hard multi-hop with distractors):
Question: Which company employs the creator of Python?

Documents:
3. Python was created by Guido van Rossum.
9. Guido van Rossum joined Microsoft in 2020.
17. Microsoft develops Windows.
28. Python is widely used in machine learning.
41. Guido van Rossum previously worked at Dropbox.

<thought>Need creator and current employer.</thought>
<tool_call>
{"name":"submit_answer","arguments":{"ids":[3,9]}}
</tool_call>


EXAMPLE 4 (implicit bridge reasoning):
Question: Which space agency launched the spacecraft named after the astronomer who discovered four moons of Jupiter?

Documents:
5. Galileo Galilei discovered four moons orbiting Jupiter.
13. The Galileo spacecraft was launched by NASA.
22. NASA is the U.S. space agency.
37. The James Webb Space Telescope was launched in 2021.

<thought>Need astronomer and spacecraft agency link.</thought>
<tool_call>
{"name":"submit_answer","arguments":{"ids":[5,13]}}
</tool_call>


EXAMPLE 5 (3-hop reasoning):
Question: Which company owns the operating system developed by the company founded by Bill Gates?

Documents:
6. Bill Gates co-founded Microsoft.
15. Microsoft developed Windows.
24. Windows is owned by Microsoft.
31. Satya Nadella is Microsoft's CEO.
42. Android is developed by Google.

<thought>Need founder-company-product ownership chain.</thought>
<tool_call>
{"name":"submit_answer","arguments":{"ids":[6,24]}}
</tool_call>


CRITICAL:
- Tool call is mandatory.
- Return exactly one tool call.
- Do not continue generation after the tool call.
- Never produce analysis paragraphs.
- Never output markdown.
- Never output prose outside <thought>.
"""


In [16]:
def index_search_tool(query, last, page):
    result = retriever.get_top_k_relevant(query, k=5, page = page) #retriever(query, last)   # result['docs'] — список строк документов
    
    tool_content = "\n".join(result['docs']) 
    return result['ids'], tool_content

In [17]:
llm = QwenAgent(model_name="Qwen3.5-9B") #Qwen3.5-4B

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

In [18]:
def run_agent_for_subquestion(agent, subquestion, initial_docs, ids, correct_ids, max_calls=5, max_steps=5, entropy_threshold=0.0):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"question: {subquestion}\n\nDocuments:\n{initial_docs}"}
    ]
    tool_calls_used = 0
    step_info_history = []
    final_answer_id = None
    success = False
    
    tool_call, response, metric = agent.step(messages, metric='entropy')
    

    # --- Normal processing (no intervention) ---
    if not tool_call:
        print(response)
        print("No valid tool call found. Stopping.")
        step_info = {
        "action": f"-",
        "has_correct": None,
        "entropy": metric,
        "text": response,
        }
        return [], step_info_history, False
    print(tool_call)
    name = tool_call.get("name")
    print(name)
    args = tool_call.get("arguments", {})
    print(args)

    has_correct = len(set(ids) & correct_ids) > 0

    step_info = {
        "action": f"{name}({json.dumps(args)})",
        "has_correct": has_correct,
        "entropy": metric,
        "text": response,
    }
    step_info_history.append(step_info)

    messages.append({"role": "assistant", "content": response})

    if name == "submit_answer":
        final_answer_id = args.get("ids")
        
        success = True
    else:
        print(f"Unknown tool: {name}")

    if final_answer_id is None:
        final_answer_id = 'No answer'
    return final_answer_id, step_info_history, success

In [21]:
from tqdm import tqdm
import json

unique_main_ids = questions_df['main_question_id'].unique()
results_by_main = {}
SAVE_PATH = 'results_qwen_9B_qwen.json'

for main_id in tqdm(unique_main_ids, desc="Main questions"):
    main_question = main_question_df[main_question_df['id'] == main_id]['question'].values[0]
    correct_ids = set(questions_df[questions_df['main_question_id'] == main_id]['paragraph_absolute_id'])
    
    # top-30 
    retrieved = retriever.get_top_k_relevant(main_question, k=30, page=0)
    retrieved_ids = retrieved['ids']           
    retrieved_docs = retrieved['docs']        
    docs_str = "\n".join(retrieved_docs)
    
    
    found_ids, step_info_history, success = run_agent_for_subquestion(
        llm,
        main_question,
        docs_str,
        retrieved_ids,
        correct_ids,
        max_calls=0,
        entropy_threshold=0
    )
    #found_ids
    if not isinstance(found_ids, list):
        found_ids = [found_ids] if found_ids else []

    found_ids = [int(x) for x in found_ids if str(x).isdigit()]
    
  
    retrieved_correct = [pid for pid in retrieved_ids if pid in correct_ids]
    num_retrieved_correct = len(retrieved_correct)
    
    found_correct = [pid for pid in found_ids if pid in correct_ids]
    num_found_correct = len(found_correct)
    
    results_by_main[main_id] = {
        "main_question": main_question,
        "retrieved_ids": retrieved_ids,
        "retrieved_correct_ids": retrieved_correct,
        "num_retrieved_correct": num_retrieved_correct,
        "total_correct_ids": list(correct_ids),
        "total_correct_count": len(correct_ids),
        "found_ids": found_ids,
        "found_correct_ids": found_correct,
        "num_found_correct": num_found_correct,
        "step_info_history": step_info_history,
        "success": success
    }
    
    with open(SAVE_PATH, 'w', encoding='utf-8') as f:
        json.dump(results_by_main, f, ensure_ascii=False, indent=2, default=str)

print(f"Результаты сохранены в {SAVE_PATH}")

Main questions:   0%|          | 0/100 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Main questions:   1%|          | 1/100 [00:21<34:54, 21.16s/it][transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


{'name': 'submit_answer', 'arguments': {'ids': [1, 11]}}
submit_answer
{'ids': [1, 11]}


Main questions:   2%|▏         | 2/100 [00:38<30:58, 18.97s/it][transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


{'name': 'submit_answer', 'arguments': {'ids': [29, 40]}}
submit_answer
{'ids': [29, 40]}


Main questions:   3%|▎         | 3/100 [01:22<48:44, 30.15s/it][transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

{'name': 'submit_answer', 'arguments': {'ids': [1947, 1957, 1949]}}
submit_answer
{'ids': [1947, 1957, 1949]}


Main questions:  99%|█████████▉| 99/100 [1:06:29<00:40, 40.02s/it][transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


{'name': 'submit_answer', 'arguments': {'ids': []}}
submit_answer
{'ids': []}


Main questions: 100%|██████████| 100/100 [1:08:24<00:00, 41.04s/it]

The user is asking about the original voice of the character that Mila Kunis plays on Family Guy.

From the documents:
- Document 1982: "Mila Kunis voices Meg Griffin. Kunis won the role after auditions... MacFarlane called Kunis back after her first audition... Once she claimed that she had it under control, MacFarlane hired her."
- Document 1996: "Meg Griffin Family Guy character... Voiced by Lacey Chabert (1999 -- 2000, 2011, 2012) Mila Kunis (1999 -- present)..."
- Document 1985: "Originally voiced by Lacey Chabert during the first season, she has been voiced by Mila Kunis since season 2..."
- Document 1995: "Lacey Chabert voiced Meg for the first production season (14 episodes)..."

So the character is Meg Griffin, and the original voice was Lacey Chabert (first season/1999-2000).

The question asks "Who did the original voice of the character Mila Kunis plays on Family Guy?"

The answer is Lacey Chabert.

I need to select the documents that provide this information. Documents 198

In [ ]:
import json

with open('results_qwen_9B_qwen.json', 'r', encoding='utf-8') as f:
    results_by_main = json.load(f)

In [ ]:
total_questions = len(results_by_main)
retriever_all_found = 0
model_all_found = 0

total_retrieved_correct_sum = 0
total_found_correct_sum = 0
total_correct_count_sum = 0

for main_id, data in results_by_main.items():
    total_correct = data['total_correct_count']
    retrieved_correct = data['num_retrieved_correct']
    found_correct = data['num_found_correct']
    
    total_correct_count_sum += total_correct
    total_retrieved_correct_sum += retrieved_correct
    total_found_correct_sum += found_correct
    
    if retrieved_correct == total_correct:
        retriever_all_found += 1
    if found_correct == total_correct:
        model_all_found += 1

retriever_perfect = retriever_all_found / total_questions * 100
model_perfect = model_all_found / total_questions * 100

avg_retrieved_correct = total_retrieved_correct_sum / total_questions
avg_found_correct = total_found_correct_sum / total_questions
avg_total_correct = total_correct_count_sum / total_questions

print("=== METRICS ===")
print(f"Total questions: {total_questions}")
print(f"Retriever (top-30) found all correct paragraphs in {retriever_all_found} questions ({retriever_perfect:.2f}%)")
print(f"Model (LLM) found all correct paragraphs in {model_all_found} questions ({model_perfect:.2f}%)")
print(f"Average number of correct paragraphs per question: {avg_total_correct:.2f}")
print(f"Average number of correct paragraphs retrieved by retriever: {avg_retrieved_correct:.2f}")
print(f"Average number of correct paragraphs found by model: {avg_found_correct:.2f}")

recall_retriever = total_retrieved_correct_sum / total_correct_count_sum * 100
recall_model = total_found_correct_sum / total_correct_count_sum * 100
print(f"Retriever recall: {recall_retriever:.2f}%")
print(f"Model recall: {recall_model:.2f}%")